# Create zarr stroring unstitched tiles

Ome-zarr libs:
https://github.com/ome/ome-zarr-py/issues/407

In [ ]:
import glob
import os.path

from ngff_zarr import to_ngff_image, to_multiscales, to_ngff_zarr
import numpy as np
import tifffile
import zarr

## Source data

In [ ]:
source_path = 'C:/Project/slides/example_cells/*.tiff'

filenames = glob.glob(source_path)
filenames

## Store as zarr

In [ ]:
pyramid_downsample = 2
npyramid_add = 2
zarr_version = 3
ome_version = '0.6'
pixel_size = {'y': 0.004, 'x': 0.004}
pixel_size_list = np.array(list(pixel_size.values()))
output_path = '../output/tiles.ome.zarr'
dim_order='cyx'


def read_tiff(filename):
    tif = tifffile.TiffFile(filename)
    data = tif.asarray()
    data = np.moveaxis(data, -1, 0)
    metadata = {}
    return data, metadata


def write_root(path):
    root = zarr.create_group(store=path, zarr_format=zarr_version, overwrite=True)
    return root


def write_tile(path, data, dim_order, pixel_size, translation=None):
    axes_units = {dim: 'micrometer' for dim in dim_order if dim in 'xyz'}
    image = to_ngff_image(data, dims=dim_order, scale=pixel_size, name='',
                          translation=translation, axes_units=axes_units)

    scale_factors = [pyramid_downsample ** (scale + 1) for scale in range(npyramid_add)]
    multiscales = to_multiscales(image, scale_factors=scale_factors, chunks=1024)

    chunks_per_shard = None

    to_ngff_zarr(path, multiscales, chunks_per_shard=chunks_per_shard, version=ome_version)

    return multiscales.metadata


def create_coordinate_transform(path, translation):
    if isinstance(translation, np.ndarray):
        translation = translation.tolist()
    return {
        "input": path,
        "output": "stitched",
        "type": "sequence",
        "transformations": [
            {
              "name": "affine_identity",
              "type": "affine",
              "affine": [
                  [1, 0, 0],
                  [0, 1, 0],
                  [0, 0, 1]
              ],
            },
            {
              "name": "translate_tile_to_stitched_position",
              "type": "translation",
              "translation": translation
            }
        ]
    }


def write_root_metadata(root, coordinate_transforms):
    root.attrs["ome"] = {"coordinateTransformations": coordinate_transforms}

## Store as zarr

In [ ]:
root = write_root(output_path)

coordinate_transforms = []
for filename in filenames[:2]:
    data, metadata = read_tiff(filename)
    tile_name = os.path.splitext(os.path.basename(filename))[0]
    path = f'{output_path}/{tile_name}'
    print(f'{filename} -> {path}', data.shape)
    metadata = write_tile(path, data, dim_order=dim_order, pixel_size=pixel_size)
    translation = np.array(list(map(int, tile_name.split('_')[:2]))) * data.shape[1:] * pixel_size_list
    coordinate_transforms.append(create_coordinate_transform(tile_name, translation))

write_root_metadata(root, coordinate_transforms)